# Thesis Results Visualization

This notebook consolidates the final visuals needed for the thesis.

It covers:
- supervised model performance for RQ1
- reasoning benchmark summaries for RQ2
- interpretive proxy summaries for RQ3
- simple workflow diagrams for methodology chapters
- optional figure export to `results/figures/`


In [ ]:
from __future__ import annotations

import io
import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 220

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = REPO_ROOT / "results"
PROMPT_EVAL_DIR = RESULTS_DIR / "prompt_eval"
FIGURES_DIR = RESULTS_DIR / "figures"
EXPORT_FIGURES = True

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print("REPO_ROOT:", REPO_ROOT)
print("FIGURES_DIR:", FIGURES_DIR)


In [ ]:
def save_current_figure(stem: str) -> None:
    if not EXPORT_FIGURES:
        return
    path = FIGURES_DIR / f"{stem}.png"
    plt.savefig(path, bbox_inches="tight")
    print("Saved:", path.relative_to(REPO_ROOT))


def clean_label(value: str) -> str:
    return str(value).replace("_", " ").replace("-", " ")


def annotate_bars(ax, fmt: str = "{:.3f}") -> None:
    for patch in ax.patches:
        height = patch.get_height()
        if pd.isna(height):
            continue
        ax.annotate(
            fmt.format(height),
            (patch.get_x() + patch.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=9,
            xytext=(0, 4),
            textcoords="offset points",
        )


## Load Supervised Metrics

The metrics file currently contains duplicate runs and at least one malformed merged line. The helper below repairs obvious timestamp merges and then loads the table into pandas.

In [ ]:
def load_metrics_table(path: Path) -> pd.DataFrame:
    text = path.read_text(encoding="utf-8")
    text = re.sub(r"(?<=\d)(?=20\d{2}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})", "\n", text)
    frame = pd.read_csv(io.StringIO(text))
    frame["timestamp_utc"] = pd.to_datetime(frame["timestamp_utc"], utc=True, errors="coerce")
    numeric_cols = [
        "accuracy",
        "macro_f1",
        "micro_f1",
        "balanced_accuracy",
        "mcc",
        "auroc_ovr",
        "pr_auc_macro",
        "brier_score",
        "ece_10bin",
    ]
    for col in numeric_cols:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
    return frame.sort_values("timestamp_utc").reset_index(drop=True)


metrics_all = load_metrics_table(RESULTS_DIR / "metrics.csv")
metrics_all

In [ ]:
metrics_latest = (
    metrics_all.sort_values("timestamp_utc")
    .groupby(["dataset", "model"], as_index=False)
    .tail(1)
    .sort_values(["dataset", "macro_f1"], ascending=[True, False])
    .reset_index(drop=True)
)

metrics_latest

## RQ1 Tables

These tables are useful for direct export into the thesis body or appendix.

In [ ]:
summary_cols = [
    "dataset",
    "model",
    "accuracy",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "auroc_ovr",
    "ece_10bin",
]
rq1_table = metrics_latest[summary_cols].copy()
rq1_table

In [ ]:
best_by_dataset = (
    metrics_latest.sort_values(["dataset", "macro_f1", "balanced_accuracy"], ascending=[True, False, False])
    .groupby("dataset", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
best_by_dataset

## RQ1 Performance Heatmaps

In [ ]:
for metric in ["accuracy", "macro_f1", "balanced_accuracy", "mcc"]:
    pivot = metrics_latest.pivot(index="model", columns="dataset", values=metric)
    plt.figure(figsize=(8, 5))
    sns.heatmap(pivot, annot=True, cmap="YlGnBu", vmin=0, vmax=1 if metric != "mcc" else None)
    plt.title(f"RQ1 {metric.replace('_', ' ').title()} by Model and Dataset")
    plt.xlabel("Dataset")
    plt.ylabel("Model")
    plt.tight_layout()
    save_current_figure(f"rq1_heatmap_{metric}")
    plt.show()


## RQ1 Dataset-Specific Comparison Charts

In [ ]:
for dataset in sorted(metrics_latest["dataset"].unique()):
    subset = metrics_latest[metrics_latest["dataset"] == dataset].sort_values("macro_f1", ascending=False)
    plt.figure(figsize=(10, 5))
    ax = sns.barplot(data=subset, x="model", y="macro_f1", color="#4C78A8")
    annotate_bars(ax)
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha="right")
    plt.title(f"RQ1 Macro F1 on {dataset.upper()}")
    plt.xlabel("Model")
    plt.ylabel("Macro F1")
    plt.tight_layout()
    save_current_figure(f"rq1_{dataset}_macro_f1")
    plt.show()


In [ ]:
ranking_plot = metrics_latest.sort_values(["dataset", "macro_f1"], ascending=[True, False]).copy()
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=ranking_plot, x="dataset", y="macro_f1", hue="model")
plt.ylim(0, 1)
plt.title("RQ1 Cross-Dataset Macro F1 Comparison")
plt.xlabel("Dataset")
plt.ylabel("Macro F1")
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_current_figure("rq1_cross_dataset_macro_f1")
plt.show()


## RQ1 Reliability and Calibration Views

In [ ]:
reliability = metrics_latest.dropna(subset=["ece_10bin", "macro_f1"]).copy()
if not reliability.empty:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=reliability, x="ece_10bin", y="macro_f1", hue="dataset", style="model", s=140)
    plt.title("RQ1 Calibration Error vs Macro F1")
    plt.xlabel("ECE (lower is better)")
    plt.ylabel("Macro F1")
    plt.tight_layout()
    save_current_figure("rq1_ece_vs_macro_f1")
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=reliability, x="brier_score", y="macro_f1", hue="dataset", style="model", s=140)
    plt.title("RQ1 Brier Score vs Macro F1")
    plt.xlabel("Brier Score (lower is better)")
    plt.ylabel("Macro F1")
    plt.tight_layout()
    save_current_figure("rq1_brier_vs_macro_f1")
    plt.show()
else:
    print("No calibration-compatible rows found.")


## RQ1 Qualitative Errors

This surfaces example errors from the saved qualitative file. The filename in the repo is currently misspelled as `ualitative_examples.txt`.

In [ ]:
qual_path = RESULTS_DIR / "ualitative_examples.txt"
if qual_path.exists():
    preview = qual_path.read_text(encoding="utf-8", errors="replace").splitlines()[:80]
    print("\n".join(preview))
else:
    print("Qualitative example file not found.")


## Prompt-Eval Run Inventory

This section combines all available prompt-eval runs under `results/prompt_eval/`.

In [ ]:
def load_prompt_eval_runs(root: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    configs = []
    model_summaries = []
    scenario_summaries = []
    item_summaries = []
    if not root.exists():
        return (pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame())
    for run_dir in sorted(path for path in root.iterdir() if path.is_dir()):
        config_path = run_dir / "config.json"
        if config_path.exists():
            config = json.loads(config_path.read_text(encoding="utf-8"))
            config["run_dir"] = str(run_dir)
            configs.append(config)
        for name, bucket in [
            ("model_summary.csv", model_summaries),
            ("scenario_scores.csv", scenario_summaries),
            ("item_scores.csv", item_summaries),
        ]:
            path = run_dir / name
            if path.exists():
                frame = pd.read_csv(path)
                frame["run_dir"] = run_dir.name
                bucket.append(frame)
    return (
        pd.DataFrame(configs),
        pd.concat(model_summaries, ignore_index=True) if model_summaries else pd.DataFrame(),
        pd.concat(scenario_summaries, ignore_index=True) if scenario_summaries else pd.DataFrame(),
        pd.concat(item_summaries, ignore_index=True) if item_summaries else pd.DataFrame(),
    )


prompt_configs, prompt_model_summary, prompt_scenarios, prompt_items = load_prompt_eval_runs(PROMPT_EVAL_DIR)
display(prompt_configs)
display(prompt_model_summary.head())


## RQ2 Reasoning Benchmark Visuals

In [ ]:
rq2_model = prompt_model_summary[prompt_model_summary["dataset"].isin(["moralbench", "morebench_public", "morebench_theory"])].copy()
if not rq2_model.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=rq2_model, x="dataset", y="primary_score", hue="model")
    plt.ylim(0, 1)
    plt.title("RQ2 Reasoning Benchmark Primary Scores")
    plt.xlabel("Benchmark")
    plt.ylabel("Primary Score")
    plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    save_current_figure("rq2_reasoning_primary_scores")
    plt.show()
else:
    print("No reasoning benchmark summaries found.")


In [ ]:
rq2_scenarios = prompt_scenarios[prompt_scenarios["dataset"].isin(["moralbench", "morebench_public", "morebench_theory"])].copy()
if not rq2_scenarios.empty and "avg_primary_score" in rq2_scenarios.columns:
    plt.figure(figsize=(12, 5))
    sns.barplot(data=rq2_scenarios, x="scenario_group", y="avg_primary_score", hue="dataset")
    plt.xticks(rotation=35, ha="right")
    plt.ylim(0, 1)
    plt.title("RQ2 Scenario-Level Scores")
    plt.xlabel("Scenario Group")
    plt.ylabel("Average Primary Score")
    plt.tight_layout()
    save_current_figure("rq2_scenario_scores")
    plt.show()
else:
    print("No reasoning scenario summaries found.")


In [ ]:
if not prompt_items.empty and "rubric_dimension_coverage" in prompt_items.columns:
    rq2_rubric = prompt_items[prompt_items["dataset"].isin(["morebench_public", "morebench_theory"])].copy()
    if not rq2_rubric.empty:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=rq2_rubric, x="dataset", y="rubric_dimension_coverage", hue="model")
        plt.title("RQ2 Rubric Dimension Coverage")
        plt.xlabel("Benchmark")
        plt.ylabel("Rubric Dimension Coverage")
        plt.tight_layout()
        save_current_figure("rq2_rubric_dimension_coverage")
        plt.show()
else:
    print("No rubric coverage rows found for RQ2.")


## RQ3 Interpretive Proxy Visuals

In [ ]:
rq3_model = prompt_model_summary[prompt_model_summary["dataset"] == "interpretive"].copy()
if not rq3_model.empty:
    pivot = rq3_model.pivot(index="model", columns="metric_id", values="primary_score")
    plt.figure(figsize=(10, 4))
    sns.heatmap(pivot, annot=True, cmap="YlGnBu", vmin=0, vmax=1)
    plt.title("RQ3 Interpretive Proxy Scores by Model")
    plt.xlabel("Proxy Metric")
    plt.ylabel("Model")
    plt.tight_layout()
    save_current_figure("rq3_proxy_metric_heatmap")
    plt.show()
else:
    print("No interpretive model summaries found.")


In [ ]:
rq3_scenarios = prompt_scenarios[prompt_scenarios["dataset"] == "interpretive"].copy()
if not rq3_scenarios.empty and "group_consistency_score" in rq3_scenarios.columns:
    plot_frame = rq3_scenarios.dropna(subset=["group_consistency_score"]).copy()
    if not plot_frame.empty:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=plot_frame, x="metric_id", y="group_consistency_score", hue="model")
        plt.ylim(0, 1)
        plt.title("RQ3 Scenario Consistency Scores")
        plt.xlabel("Metric")
        plt.ylabel("Group Consistency Score")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        save_current_figure("rq3_group_consistency")
        plt.show()
else:
    print("No interpretive consistency rows found.")


In [ ]:
if not prompt_items.empty and {"confidence_0_100", "answer_correct"}.issubset(prompt_items.columns):
    rq3_conf = prompt_items[prompt_items["dataset"] == "interpretive"].dropna(subset=["confidence_0_100", "answer_correct"]).copy()
    if not rq3_conf.empty:
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=rq3_conf, x="confidence_0_100", y="answer_correct", hue="model", style="metric_id", s=120)
        plt.title("RQ3 Confidence vs Correctness")
        plt.xlabel("Confidence (0-100)")
        plt.ylabel("Correctness")
        plt.tight_layout()
        save_current_figure("rq3_confidence_vs_correctness")
        plt.show()
else:
    print("No confidence/correctness rows found for interpretive prompts.")


## Workflow and Method Diagrams

These are deliberately simple diagram figures that can be exported directly into the methodology chapter.

In [ ]:
def draw_box(ax, x, y, w, h, text, color="#E8F1FA"):
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="#2F4B7C", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=11, wrap=True)


def arrow(ax, x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=1.5, color="#444444"))


fig, ax = plt.subplots(figsize=(12, 3.8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)
ax.axis("off")
draw_box(ax, 0.3, 1.3, 2.0, 1.1, "Raw Data\nData/raw")
draw_box(ax, 2.7, 1.3, 2.0, 1.1, "Preprocess + EDA")
draw_box(ax, 5.1, 1.3, 2.1, 1.1, "Benchmark Layers\nSupervised / Reasoning / Interpretive")
draw_box(ax, 7.7, 1.3, 1.8, 1.1, "Run Experiments")
draw_box(ax, 9.9, 1.3, 1.7, 1.1, "Results / Figures")
arrow(ax, 2.3, 1.85, 2.7, 1.85)
arrow(ax, 4.7, 1.85, 5.1, 1.85)
arrow(ax, 7.2, 1.85, 7.7, 1.85)
arrow(ax, 9.5, 1.85, 9.9, 1.85)
plt.title("End-to-End Project Workflow")
plt.tight_layout()
save_current_figure("workflow_end_to_end")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.set_xlim(0, 11)
ax.set_ylim(0, 4)
ax.axis("off")
draw_box(ax, 0.3, 1.3, 2.0, 1.1, "Benchmark CSV")
draw_box(ax, 2.8, 1.3, 2.1, 1.1, "run\nresponses.jsonl", color="#FFF2CC")
draw_box(ax, 5.4, 1.3, 2.0, 1.1, "score\nitem_scores.csv", color="#FFF2CC")
draw_box(ax, 7.9, 1.3, 2.3, 1.1, "aggregate\nscenario_scores + model_summary", color="#FFF2CC")
arrow(ax, 2.3, 1.85, 2.8, 1.85)
arrow(ax, 4.9, 1.85, 5.4, 1.85)
arrow(ax, 7.4, 1.85, 7.9, 1.85)
plt.title("Prompt Evaluation Pipeline")
plt.tight_layout()
save_current_figure("workflow_prompt_eval")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.set_xlim(0, 11)
ax.set_ylim(0, 4)
ax.axis("off")
draw_box(ax, 0.3, 2.2, 2.5, 1.0, "Supervised Layer\nETHICS / NormBank / MFRC", color="#D9EAD3")
draw_box(ax, 4.1, 2.2, 2.5, 1.0, "Reasoning Layer\nMoralBench / MoReBench", color="#D9EAF7")
draw_box(ax, 7.9, 2.2, 2.5, 1.0, "Interpretive Layer\nProxy Metrics", color="#F4D9E8")
draw_box(ax, 2.2, 0.6, 2.3, 0.9, "Classification Metrics", color="#FCE5CD")
draw_box(ax, 4.5, 0.6, 2.0, 0.9, "Rubric / Heuristic Scores", color="#FCE5CD")
draw_box(ax, 6.8, 0.6, 2.2, 0.9, "Proxy / Consistency Scores", color="#FCE5CD")
arrow(ax, 1.55, 2.2, 3.35, 1.5)
arrow(ax, 5.35, 2.2, 5.5, 1.5)
arrow(ax, 9.15, 2.2, 7.9, 1.5)
plt.title("Three-Layer Evaluation Design")
plt.tight_layout()
save_current_figure("workflow_three_layer_design")
plt.show()


## Final Figure Inventory

In [ ]:
figure_files = sorted(FIGURES_DIR.glob("*.png"))
pd.DataFrame({
    "figure": [path.name for path in figure_files],
    "size_kb": [round(path.stat().st_size / 1024, 1) for path in figure_files],
})
